# PASTORS + TabPFN (Google Colab)

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

Upload to a Drive folder (e.g. `MyDrive/pastors/`) keeping this structure:
```
pastors/
  helpers/                         (all .py)
  data/geospatial/oman_processed_closed.geojson
  data/geospatial/validation_processed_closed.geojson
  data/satellite/fused_spot_s2_oman.tif (+ .json + _metadata.json)
  output/baresoil_mask_oman.tif        (only for raster prediction)
```
Set `BASE` below to that folder.

In [1]:
!pip install -q tabpfn-client rasterio geopandas composition_stats

In [2]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/pastors'     # <-- your Drive folder
os.chdir(BASE); sys.path.insert(0, BASE)

# --- TabPFN cloud client: load token from local file (git-ignored) ---
import json as _json
from tabpfn_client import set_access_token
with open('./utils/tabpfn_token.json') as _f:
    _tok = _json.load(_f)['token']
set_access_token(_tok)
print("TabPFN client authenticated")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TabPFN client authenticated


In [ ]:
import os
import numpy as np, pandas as pd, geopandas as gpd
from helpers.prepare_validation import prepare_validation
from helpers.background_removal import background_removal
from helpers.load_satellite_data import load_satellite_data
from helpers.symmetric_balances import group_balance
from helpers.dendro_groups import compositional_groups
from helpers.spatial_cv import (make_spatial_groups, spatial_cv_predict,
                                regression_metrics, agreement_metrics,
                                resolve_threshold)
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from tabpfn_client import TabPFNRegressor

# ── Config ──
dataset   = "kenya" #"kenya", "oman"
satellite = f"fused_spot_s2_{dataset}"
buffer    = 500
SHARPEN_SWIR          = False
RESTRICT_TRAINING     = True
USE_GLOBAL_BACKGROUND = False
AUTO_GROUPS       = True
if dataset == "oman":
    ACTIVE_SITES        = ["S4","S7","S10"]
    GROUP_EXCLUDE_SITES = ["S11","S12"]
    GROUP_TYPE_PREFIX   = "CORRAL"
else:  # kenya
    ACTIVE_SITES        = ["E2","E4","E7","E9"]   # pick your enclosure sites
    GROUP_EXCLUDE_SITES = []
    GROUP_TYPE_PREFIX   = "COR"   # matches both CORAL and CORRAL

# fallback manual groups if AUTO_GROUPS=False
MARKERS_MANUAL    = ["P","K","Ca"]
NATBG_MANUAL      = ["Si","Al","Fe","Ti","Mg"]

# ── Evaluation parity with pastors.ipynb (XGBoost) ──
# Score TabPFN with the SAME leave-site-out spatial CV, threshold and metrics
# so the two models are directly comparable (not a leaky random split).
AGREEMENT_THRESHOLD = 0.0     # depleted (<thr) / elevated (>thr); 0.0 or "antimode"
CV_MAX_FOLDS = None           # None = leave-site-out (exact parity); int = GroupKFold(n)
                              #   -> set e.g. 5 to cut TabPFN cloud calls (loses split parity)

In [4]:
# ── Load, background-remove, detect groups, build balance ──
gdf  = gpd.read_file(f"./data/geospatial/{dataset}_processed_closed.geojson")

VAL_PATH = "./data/geospatial/validation_processed_closed.geojson"
HAS_VAL  = (dataset == "oman") and os.path.exists(VAL_PATH)

if HAS_VAL:
    gval = gpd.read_file(VAL_PATH)
    gval["Serial"] = gval["Serial"].astype(str)
    gdf, gval_bg, shared = prepare_validation(gdf, gval, restrict_training=RESTRICT_TRAINING)
else:
    gval_bg = None
    print("No validation set — skipping external validation")

if AUTO_GROUPS:
    grp = compositional_groups(f"./data/geospatial/{dataset}_processed_closed.geojson",
            include_sites=ACTIVE_SITES, exclude_sites=GROUP_EXCLUDE_SITES,
            type_prefix=GROUP_TYPE_PREFIX, exclude_parts=("Res",))
    markers, natural_bgk = grp["markers"], grp["natural_bgk"]
else:
    markers, natural_bgk = MARKERS_MANUAL, NATBG_MANUAL
print("markers:", markers, "| natural_bgk:", natural_bgk)

gbk = background_removal(gdf, buffer_distance=buffer, use_global_background=USE_GLOBAL_BACKGROUND)
df  = load_satellite_data(satellite, gbk, sharpen_swir=SHARPEN_SWIR)
df["balance"] = group_balance(df, markers, natural_bgk)

if HAS_VAL:
    gval_bk = background_removal(gval_bg, buffer_distance=buffer, use_global_background=USE_GLOBAL_BACKGROUND)
    gval_bk = gval_bk.loc[gval_bk.index.intersection(gval_bg["Serial"])]
    dfv = load_satellite_data(satellite, gval_bk, sharpen_swir=SHARPEN_SWIR)
    dfv["balance"] = group_balance(dfv, markers, natural_bgk)

No validation set — skipping external validation
markers: ['Mg', 'P', 'K', 'Ca', 'Mn', 'Zn', 'Rb', 'Sr', 'Zr'] | natural_bgk: ['Al', 'Si', 'Ti', 'Fe', 'Y', 'Ba']
✅ Background removal complete for 20 sites.
Loading raster: ./data/satellite/fused_spot_s2_kenya.tif
  aggregated 23 multi-sample pixels
  added NDVI, SAVI
  added BSI, NDMI, Clay_Index
  added NDWI, Calcite_Index
  output: 1603 rows x 37 cols (16 elements + 14 bands + 7 indices)
✅ Satellite data integration complete
✅ Completed balance comparing between marker-elements and natural-background.


In [ ]:
# ── Feature matrix ──
# 'Emb_' included for parity with pastors.ipynb (harmless if no embedding bands).
sat_kw = ("Band_","Drag_","S2_","SPOT_","Maxar_","Emb_")
idx_kw = ("NDVI","SAVI","BSI","NDMI","NDWI","MSAVI","Clay","Calcite","Iron","EVI","NBR")
feature_cols = [c for c in df.columns if c.startswith(sat_kw) or any(c.startswith(k) for k in idx_kw)]
X = df[feature_cols].values; y = df["balance"].values

# Site label per pixel -> leave-site-out spatial CV groups (same as XGBoost).
df["Site"] = gdf.set_index("Serial").loc[df.index, "Site"].values
groups_all, _ = make_spatial_groups(sites=df["Site"].values, strategy="site")

print(f"features: {len(feature_cols)} | train samples: {len(y)} | "
      f"sites: {len(np.unique(groups_all))} | val: {len(dfv) if HAS_VAL else 0}")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
reg = TabPFNRegressor(n_estimators=8)
reg.fit(X_tr, y_tr)

def metrics(yt, yp):
    return dict(R2=round(r2_score(yt,yp),3),
                RMSE=round(np.sqrt(mean_squared_error(yt,yp)),3),
                MAE=round(mean_absolute_error(yt,yp),3),
                Pearson=round(pearsonr(yt,yp)[0],3),
                Spearman=round(spearmanr(yt,yp)[0],3))
print("Internal 20% test:", metrics(y_te, reg.predict(X_te)))

00:09 Fitting... Done!
00:08 Predicting... Done!
Internal 20% test: {'R2': 0.311, 'RMSE': np.float64(0.229), 'MAE': 0.127, 'Pearson': np.float64(0.559), 'Spearman': np.float64(0.528)}


In [ ]:
# ── Apples-to-apples vs XGBoost: leave-site-out spatial CV + agreement ──────
# SAME splitter, threshold and metrics as pastors.ipynb, but TabPFN via model_fn.
# NOTE: fits TabPFN once per held-out site (cloud calls) — slow. Set
# CV_MAX_FOLDS=5 in the config to cut calls (loses exact split parity).
oof_tab, foldr2_tab, cvname = spatial_cv_predict(
    X, y, groups_all,
    model_fn=lambda: TabPFNRegressor(n_estimators=8),
    max_folds=CV_MAX_FOLDS,
)

vmask = np.isfinite(oof_tab)
thr   = resolve_threshold(AGREEMENT_THRESHOLD, y)
reg_m = regression_metrics(y[vmask], oof_tab[vmask])
agr_m = agreement_metrics(y[vmask], oof_tab[vmask], threshold=thr)

print(f"TabPFN spatial CV ({cvname}) — n={int(vmask.sum())}, thr={thr:.3f}")
for k in ["R2", "RMSE", "MAE", "Pearson", "Spearman"]:
    print(f"  {k:9s}: {reg_m[k]:.3f}")
for k in ["AUC", "balanced_acc", "MCC", "kappa", "F1", "accuracy"]:
    print(f"  {k:12s}: {agr_m[k]:.3f}")
if foldr2_tab:
    print(f"  per-fold R²: mean={np.mean(foldr2_tab):.3f} std={np.std(foldr2_tab):.3f}")

# Row schema matches the XGBoost agreement CSV (metrics columns align), with a
# 'model' column so the two can be concatenated for the comparison table.
row = {"model": "TabPFN", "eval": "spatial_cv", "dataset": dataset,
       "cv": cvname, "thr": round(thr, 3),
       "R2": round(reg_m["R2"], 3), "Spearman": round(reg_m["Spearman"], 3),
       "AUC": round(agr_m["AUC"], 3), "bal_acc": round(agr_m["balanced_acc"], 3),
       "MCC": round(agr_m["MCC"], 3)}
pd.DataFrame([row]).to_csv(f"./output/{dataset}_tabpfn_metrics.csv", index=False)
print("saved -> ./output/{}_tabpfn_metrics.csv".format(dataset))
pd.DataFrame([row])

In [ ]:
# ── External validation — same two figures as pastors.ipynb (XGBoost) ───────
# Fig 1: predicted vs true, coloured by residual, with the stats box.
# Fig 2: elevated/depleted agreement (correct/wrong side of the threshold).
# NOTE: XGBoost samples the prediction RASTER at the validation points, whereas
# TabPFN predicts DIRECTLY on the validation features here — so the plots look
# the same but the point sets aren't guaranteed identical. Use the spatial-CV
# numbers for the strict model comparison.
EXCLUDE_VAL_IDS = [3, 5]   # drop mis-sampled validation points (parity with XGBoost)

reg_full = TabPFNRegressor(n_estimators=8).fit(X, y)

if HAS_VAL:
    import matplotlib.pyplot as plt
    Xv = dfv[feature_cols].values
    yv = dfv["balance"].values
    pred_v = reg_full.predict(Xv)

    df_eval = pd.DataFrame({"true": yv, "predicted": pred_v})
    df_eval = df_eval.replace([np.inf, -np.inf], np.nan).dropna()
    df_eval["residual"] = df_eval["predicted"] - df_eval["true"]
    if EXCLUDE_VAL_IDS:
        drop = [i for i in EXCLUDE_VAL_IDS if i in df_eval.index]
        if drop:
            print(f"  excluding validation ids {drop} (mis-sampled)")
            df_eval = df_eval.drop(index=drop)
    eval_label = "External validation"

    yt = df_eval["true"].values
    yp = df_eval["predicted"].values
    res = df_eval["residual"].values
    r2   = r2_score(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    pear = pearsonr(yt, yp)[0]
    spear = spearmanr(yt, yp)[0]

    # ---- Figure 1: predicted vs true, coloured by residual ----
    fig, ax = plt.subplots(figsize=(9, 8))
    vmax = max(abs(res.min()), abs(res.max()))
    sc = ax.scatter(yt, yp, c=res, cmap="RdYlBu_r", vmin=-vmax, vmax=vmax,
                    s=60, alpha=0.85, edgecolor="black", linewidth=0.5, zorder=3)
    cb = plt.colorbar(sc); cb.set_label("Residual (Pred − True)", rotation=270, labelpad=15)
    if len(df_eval) <= 80:
        for idx, r in df_eval.iterrows():
            ax.text(r["true"] + 0.015, r["predicted"] + 0.015, str(idx),
                    fontsize=8, alpha=0.8, zorder=4)
    lim = [min(yt.min(), yp.min()) - 0.1, max(yt.max(), yp.max()) + 0.1]
    ax.plot(lim, lim, "r--", lw=1.5, label="Ideal (1:1)", zorder=2)
    stats = (f"n = {len(df_eval)}\n$R^2$ = {r2:.3f}\nRMSE = {rmse:.3f}\n"
             f"MAE = {mae:.3f}\nPearson r = {pear:.3f}\nSpearman ρ = {spear:.3f}")
    ax.text(0.05, 0.95, stats, transform=ax.transAxes, fontsize=11, va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))
    ax.set_xlabel("Measured/True Balance", fontsize=12, fontweight="bold")
    ax.set_ylabel("Predicted Balance", fontsize=12, fontweight="bold")
    ax.set_title(f"{dataset.title()} — {eval_label} (TabPFN): Predicted vs True",
                 fontsize=14, pad=15)
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"./output/{dataset}_external_validation_pred_vs_true_tabpfn.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ---- Figure 2: elevated/depleted agreement (threshold from TRAINING y) ----
    thr = resolve_threshold(AGREEMENT_THRESHOLD, y)
    agr = agreement_metrics(yt, yp, threshold=thr)
    ok  = (yt > thr) == (yp > thr)
    fig, ax = plt.subplots(figsize=(7.5, 7))
    ax.axhline(thr, color="grey", ls=":", lw=1); ax.axvline(thr, color="grey", ls=":", lw=1)
    ax.scatter(yt[ok],  yp[ok],  c="seagreen", s=55, edgecolor="k", lw=.4,
               label="correct side", zorder=3)
    ax.scatter(yt[~ok], yp[~ok], c="crimson",  s=55, edgecolor="k", lw=.4,
               label="wrong side", zorder=3)
    ax.plot(lim, lim, "r--", lw=1.2, zorder=2)
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Measured balance"); ax.set_ylabel("Predicted balance")
    ax.set_title(f"{dataset.title()} — {eval_label} (TabPFN): elevated/depleted agreement\n"
                 f"AUC={agr['AUC']:.2f}  bal.acc={agr['balanced_acc']:.2f}  MCC={agr['MCC']:.2f}")
    ax.legend(loc="lower right"); plt.tight_layout()
    plt.savefig(f"./output/{dataset}_external_validation_agreement_tabpfn.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    print(f"External validation (TabPFN): {metrics(yt, yp)}")
    print(f"  agreement (thr={thr:.3f}): AUC={agr['AUC']:.3f} "
          f"bal_acc={agr['balanced_acc']:.3f} MCC={agr['MCC']:.3f}")

    # Append an 'External validation' row to the TabPFN metrics CSV (idempotent:
    # replaces any previous external row), schema matching the spatial_cv row.
    ext_row = {"model": "TabPFN", "eval": "External validation", "dataset": dataset,
               "cv": "external", "thr": round(thr, 3),
               "R2": round(r2, 3), "Spearman": round(spear, 3),
               "AUC": round(agr["AUC"], 3), "bal_acc": round(agr["balanced_acc"], 3),
               "MCC": round(agr["MCC"], 3)}
    _p = f"./output/{dataset}_tabpfn_metrics.csv"
    _prev = pd.read_csv(_p) if os.path.exists(_p) else pd.DataFrame()
    if not _prev.empty and "eval" in _prev.columns:
        _prev = _prev[_prev["eval"] != "External validation"]
    pd.concat([_prev, pd.DataFrame([ext_row])], ignore_index=True).to_csv(_p, index=False)
    print("saved figures + appended external row ->", _p)
else:
    print("No validation set — reg_full fitted on all training data for raster prediction")

In [ ]:
import rasterio
from helpers.load_satellite_data import _add_spectral_indices, _read_band_names

def predict_raster_tabpfn(reg, satellite, feature_cols, y_lo=None, y_hi=None,
                          base="./data/satellite", mask_path=None, sharpen_swir=False,
                          batch=200000, unc_pct=95, out_path=None):
    """
    Raster inference with TabPFN. Uses RAW features (the model is trained on raw
    X — no standardization anywhere), so no (x-mu)/sd here. Predictions are
    optionally clipped to [y_lo, y_hi] to curb extrapolation, and the most
    uncertain `unc_pct`% of pixels are masked in the *_masked output.
    """
    out_path = out_path or f"./output/{dataset}_tabpfn_prediction.tif"
    with rasterio.open(f"{base}/{satellite}.tif") as src:
        prof = src.profile.copy(); img = src.read().astype("float32")
        bn = _read_band_names(satellite, base, src.count); H, W = src.height, src.width
    if sharpen_swir:
        from helpers.feature_ops import sharpen_swir_bands; img = sharpen_swir_bands(img, bn)
    B = img.shape[0]
    dfx = _add_spectral_indices(pd.DataFrame(img.reshape(B, -1).T, columns=bn), satellite)
    Xr = dfx[feature_cols].values
    valid = np.isfinite(Xr).all(1)
    if mask_path:
        with rasterio.open(mask_path) as mk:
            valid &= (mk.read(1).reshape(-1) == 1)
    idx = np.where(valid)[0]
    print(f"predicting {len(idx):,} bare-soil pixels of {Xr.shape[0]:,}")

    pred = np.full(Xr.shape[0], np.nan, "float32")
    unc  = np.full(Xr.shape[0], np.nan, "float32")
    for i in range(0, len(idx), batch):
        j = idx[i:i+batch]
        Xb = Xr[j]                                   # raw features (as trained)
        pred[j] = reg.predict(Xb)
        lo, med, hi = reg.predict(Xb, output_type="quantiles", quantiles=[0.05, 0.5, 0.95])
        unc[j] = hi - lo
        print(f"  {min(i+batch, len(idx)):,}/{len(idx):,}")

    if y_lo is not None and y_hi is not None:      # clip to training range
        pred = np.where(np.isfinite(pred), np.clip(pred, y_lo, y_hi), np.nan).astype("float32")
    thr = np.nanpercentile(unc, unc_pct)            # mask the most-uncertain pixels
    pred_masked = pred.copy(); pred_masked[unc > thr] = np.nan
    print(f"masking pixels with 90% interval width > {thr:.3f} (top {100-unc_pct:.0f}%)")

    prof.update(count=1, dtype="float32", nodata=np.nan, compress="lzw")
    os.makedirs("./output", exist_ok=True)
    def _w(arr, path):
        with rasterio.open(path, "w", **prof) as d: d.write(arr.reshape(H, W), 1)
    _w(pred, out_path)
    _w(unc, out_path.replace(".tif", "_uncertainty.tif"))
    _w(pred_masked, out_path.replace(".tif", "_masked.tif"))
    print("saved ->", out_path, "(+ _uncertainty, + _masked)")
    return out_path

# Clip predictions to the observed training-balance range (curbs wild extrapolation).
Y_LO, Y_HI = float(np.min(y)), float(np.max(y))
predict_raster_tabpfn(reg_full, satellite, feature_cols, y_lo=Y_LO, y_hi=Y_HI,
                      mask_path=f"./output/baresoil_mask_{dataset}.tif",
                      sharpen_swir=SHARPEN_SWIR)